In [1]:
import os
import sys
import numpy as np
import pandas as pd
import nibabel as nib
import nilearn
from nilearn.image import resample_img
from nilearn import plotting
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from nilearn import masking
from natsort import natsorted
import pickle

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
from scipy.stats import zscore

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import KFold
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [8]:
# z_arr = np.load("N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\data\z_series_arr_12subs.npy")
z_arr = np.load("F:\yujun\projects\AI_IAPS\Decoding\data\z_series_arr_11subs_v2.npy")


In [2]:
maskDir = 'F:\yujun\projects\data\masks'
maskFile = os.path.join(maskDir, 'MNI152_T1_2mm_brain_mask.nii.gz')
mask = nib.load(maskFile)
temp_nii_path = r'F:\yujun\projects\AI_IAPS\GLM_singletrial\\betas\Sub1\\beta_0001.nii'
temp_nii_data = nib.load(temp_nii_path)
# Resample the MNI mask to match the dimensions and voxel size of your NIfTI file
resampled_mask = resample_img(mask, target_affine=temp_nii_data.affine, target_shape=temp_nii_data.shape)
# Load the dictionary from the file
with open('../Decoding/kastner_dict.pkl', 'rb') as f:
    roi_masks = pickle.load(f)
for (k, v) in roi_masks.items():
    print(k, v.get_fdata().shape, np.count_nonzero(v.get_fdata()))

V1v (79, 95, 79) 856
V1d (79, 95, 79) 748
V2v (79, 95, 79) 727
V2d (79, 95, 79) 645
V3v (79, 95, 79) 527
V3d (79, 95, 79) 517
hV4 (79, 95, 79) 328
V3a (79, 95, 79) 554
V3b (79, 95, 79) 263
LO1 (79, 95, 79) 324
LO2 (79, 95, 79) 125
VO1 (79, 95, 79) 153
VO2 (79, 95, 79) 253
IPS (79, 95, 79) 973
PHC1 (79, 95, 79) 175
PHC2 (79, 95, 79) 165


In [4]:
subs =          ['Sub1', 'Sub2', 'Sub3', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub8', 'Sub9', 'Sub11', 'Sub12','Sub13']
selected_subs = ['Sub1', 'Sub2', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub9', 'Sub11', 'Sub12', 'Sub13']
# Create a mapping from selected_subs to their corresponding index in subs list
selected_indices = [subs.index(sub) for sub in selected_subs]


In [5]:
groups = ['pleasant', 'neutral', 'unpleasant', 'pleasantAI', 'neutralAI', 'unpleasantAI']
group_order = ['pleasant', 'neutral', 'unpleasant', 'pleasantAI', 'neutralAI', 'unpleasantAI']
stimuli = pd.read_csv("../Decoding/stimuli_600trials.csv")
stimuli
# Get the order of trials based on the 'group' column and the predefined group_order
stimuli['group_order_idx'] = stimuli['group'].apply(lambda g: group_order.index(g))
# Sort the stimuli DataFrame based on the group order
stimuli_sorted = stimuli.sort_values(by='group_order_idx').reset_index(drop=True)
sorted_indices = stimuli_sorted['OriginalOrder'].values
sorted_indices

array([570, 365, 206, 207, 208, 102,  99, 211, 398, 428, 454, 215,  93,
       202, 400, 562, 507, 505, 567, 504,  74, 233, 503,  71, 407, 500,
       573, 401, 108, 197, 394, 446, 157, 448, 384, 542, 142, 141, 445,
       138, 449, 530, 529, 170, 172, 174, 177, 126, 523, 370, 185, 186,
       519, 117, 116, 441, 367, 113, 347,  61, 151, 249, 338, 335, 268,
       270,  43,  41, 272,  39, 273, 267, 430,  19, 277, 581, 336, 419,
        30, 582,  26, 340,  18, 301, 584, 259,   4, 344, 491, 304, 262,
       312, 594,  56, 423, 328,  47, 591, 252, 435, 518, 194, 190, 290,
       318, 319, 534, 168, 372, 178, 323, 308, 293, 164, 171, 469, 162,
       464, 484, 281, 497, 243, 459, 254, 239, 501, 462, 537, 231, 265,
       489, 227, 458, 354, 282, 223, 220, 357, 218, 361, 271, 342, 210,
       209, 487, 276, 346, 280, 513, 514, 356, 427, 321,  31,  16, 586,
       122,  80, 549,  55, 409,  24, 431, 404, 563,  64, 110,  32, 109,
       403, 416, 434, 555,  37,  51, 100, 411,  48, 568,  15, 38

In [9]:
z_flat = z_arr.reshape(z_arr.shape[0],z_arr.shape[1],-1)
z_flat_sorted = z_flat[:, sorted_indices, :]

MemoryError: Unable to allocate 14.6 GiB for an array with shape (600, 11, 592895) and data type float32

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.svm import SVC
from tqdm import tqdm

# Define the comparisons you want to loop through
comparisons = [
    ('pleasant', 'neutral'),
    ('pleasantAI', 'neutralAI'),
    ('unpleasant', 'neutral'),
    ('unpleasantAI', 'neutralAI')
]

# Initialize a dictionary to store the accuracies for each comparison and ROI
accuracy_results = {}

# Get the total number of iterations for progress tracking
n_comparisons = len(comparisons)
n_rois = len(roi_masks)
n_subjects = len(selected_indices)
n_repeats = 10
n_folds = 4
total_iterations = n_comparisons * n_rois * n_subjects * n_repeats * n_folds

# Initialize the tqdm progress bar
with tqdm(total=total_iterations, desc="Decoding Progress") as pbar:
    # Loop through the comparisons
    for condition1, condition2 in comparisons:
        # print(f"Decoding {condition1} vs {condition2}...")

        # Get the indices of the conditions in the group_order
        condition1_idx = group_order.index(condition1)
        condition2_idx = group_order.index(condition2)

        accuracy_rois = []

        # Loop through each ROI
        for m, (roi_name, roi_mask) in enumerate(roi_masks.items()):
            # print(f"Decoding ROI: {roi_name}...")

            # Flatten the ROI mask and get the indices of the ROI voxels
            flat_roi_data = roi_mask.get_fdata().flatten()
            indices = np.where(flat_roi_data)[0]

            # Extract the relevant data from z_series_sorted for the current ROI
            # data = z_flat_sorted[:, :, indices]
            data = z_flat_sorted[:, :, indices]

            accuracy_subs = []

            # Loop over each subject
            for subject in selected_indices:
                # Extract trials for condition1 and condition2
                subject_data = data[subject]
                condition1_trials = subject_data[condition1_idx * 100:(condition1_idx + 1) * 100]
                condition2_trials = subject_data[condition2_idx * 100:(condition2_idx + 1) * 100]

                # Remove NaN columns
                condition1_trials = condition1_trials[:, ~np.isnan(condition1_trials[0, :])]
                condition2_trials = condition2_trials[:, ~np.isnan(condition2_trials[0, :])]

                assert condition1_trials.shape[1] >= 0.8 * len(indices), \
                    f"Voxels are less than 80% of the length of indices."

                accuracies = []
                # Perform 50 repeats of 4-fold cross-validation
                for repeat in range(n_repeats):
                    kf = KFold(n_splits=n_folds, shuffle=True, random_state=repeat)
                    condition1_indices = np.arange(condition1_trials.shape[0])
                    condition2_indices = np.arange(condition2_trials.shape[0])

                    for train_idx, test_idx in kf.split(condition1_indices, condition2_indices):
                        # Split the training and testing data
                        train_condition1 = condition1_trials[train_idx]
                        train_condition2 = condition2_trials[train_idx]
                        test_condition1 = condition1_trials[test_idx]
                        test_condition2 = condition2_trials[test_idx]

                        # Average the training trials into 3 groups
                        train_condition1_averaged = np.mean(np.array_split(train_condition1, 3, axis=0), axis=1)
                        train_condition2_averaged = np.mean(np.array_split(train_condition2, 3, axis=0), axis=1)

                        # Keep the test trials as one group
                        test_condition1_averaged = np.mean(test_condition1, axis=0, keepdims=True)
                        test_condition2_averaged = np.mean(test_condition2, axis=0, keepdims=True)

                        # Combine the training and testing data
                        train_data = np.vstack((train_condition1_averaged, train_condition2_averaged))
                        train_labels = np.array([1] * len(train_condition1_averaged) + [0] * len(train_condition2_averaged))

                        test_data = np.vstack((test_condition1_averaged, test_condition2_averaged))
                        test_labels = np.array([1] * test_condition1_averaged.shape[0] + [0] * test_condition2_averaged.shape[0])

                        # Train the SVM classifier
                        clf = SVC(kernel='linear')
                        clf.fit(train_data, train_labels)
                        accuracy = clf.score(test_data, test_labels)
                        accuracies.append(accuracy)

                        # Update progress bar after each fold
                        pbar.update(1)

                # Store the average accuracy across repeats for this subject
                accuracy_subs.append(np.mean(accuracies))

            # Store the average accuracy across subjects for this ROI
            accuracy_rois.append(np.array(accuracy_subs))

        # Store the results for the current comparison
        accuracy_results[f"{condition1}_vs_{condition2}"] = np.array(accuracy_rois)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1)  convert accuracy_results → ndarray  (4, 16, 12)
# ------------------------------------------------------------
scen_order = ['pleasant_vs_neutral',
              'unpleasant_vs_neutral',
              'pleasantAI_vs_neutralAI',
              'unpleasantAI_vs_neutralAI']

acc_list = [accuracy_results[k] for k in scen_order]   # each (16, 12)
results  = np.stack(acc_list)                          # (4, 16, 12)

# ------------------------------------------------------------
# 2)  basic labels
# ------------------------------------------------------------
try:                         # ROI tags from the mask dict, if available
    roi_labels = list(roi_masks.keys())
except NameError:
    roi_labels = [f"ROI{r+1}" for r in range(results.shape[1])]

scenario_labels = [
    "Pleasant vs Neutral",
    "Unpleasant vs Neutral",
    "PleasantAI vs NeutralAI",
    "UnpleasantAI vs NeutralAI",
]

# ------------------------------------------------------------
# 3)  single wide figure: four bar‑plots side‑by‑side
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)

for scen, ax in enumerate(axes):
    mean_subj = results[scen].mean(axis=1)        # (16,)
    std_subj  = results[scen].std(axis=1, ddof=1)

    x = np.arange(len(roi_labels))
    ax.bar(x, mean_subj, yerr=std_subj, capsize=3, color='tab:cyan')
    ax.set_xticks(x)
    ax.set_xticklabels(roi_labels, rotation=45, ha='right', fontsize=8)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1)
    if scen == 0:
        ax.set_ylabel("Accuracy")
    ax.set_title(scenario_labels[scen], fontsize=10, pad=8)

fig.suptitle("Cross‑day decoding – mean ± SD across subjects", fontsize=14, y=1.05)
fig.tight_layout(rect=[0, 0.05, 1, 0.93])
plt.show()
